# 06 — Compare models and evaluate next-week surge prediction

**Question:** can price history improve detection over simple baselines under expanding-window evaluation?

Run `python prepare.py --experiment all` first. Then **Restart Kernel and Run All** here. The same readable pipeline is fitted separately for every training fold. We compare Logistic Regression, Random Forest, XGBoost and LightGBM with two modest settings each, without SMOTE or a large search.

**Order:** declare criteria → compare 2025 folds → select model → fit through December 2025 → choose cutoff on January–March 2026 → evaluate once on April–June 2026 → save prediction inputs/results for later explanation and dashboard work.

The fixed **10% price-increase label** comes from 05. A classifier's score cutoff is a separate choice. Class weighting can distort probability calibration: saved scores are **uncalibrated model scores**, not verified event probabilities. This notebook does not establish affordability benefits or stakeholder adoption.


## 1. Imports and modest experiment settings
Install `requirements.txt` first. Helpers in `pricecatcher_modelling.py` contain the repeated input checks, preprocessing and metric formulas; the experiment flow stays here. Each model uses the full training rows. Tree counts and CPU use are bounded for a laptop. The eight candidates below are declared before any evaluation.


In [ ]:
import json
import hashlib
from pathlib import Path
from importlib import metadata, reload
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay, ConfusionMatrixDisplay

import pricecatcher_config as cfg
cfg = reload(cfg)
from pricecatcher_modelling import (
    load_experiment, check_frame, model_inputs, build_pipeline, evaluate,
    choose_cutoff, baseline_scores, grouped_metrics,
)

SEED = 42
N_JOBS = 2
TREES = 150
OUTPUT_DIR = cfg.PROJECT_ROOT / "06_models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FOLDS = ["fold_1", "fold_2", "fold_3"]
CANDIDATES = {
    "lr_c01": ("Logistic Regression", {"C": 0.1}),
    "lr_c1": ("Logistic Regression", {"C": 1.0}),
    "rf_depth8": ("Random Forest", {"max_depth": 8}),
    "rf_depth16": ("Random Forest", {"max_depth": 16}),
    "xgb_depth3": ("XGBoost", {"max_depth": 3}),
    "xgb_depth5": ("XGBoost", {"max_depth": 5}),
    "lgb_leaves15": ("LightGBM", {"num_leaves": 15}),
    "lgb_leaves31": ("LightGBM", {"num_leaves": 31}),
}
print("Candidates:", len(CANDIDATES), "| Tree iterations:", TREES, "| Outputs:", OUTPUT_DIR)


## 2. Declare metrics and success criteria before training

**Primary selection metric:** mean **average precision (AP)** across the three expanding validation folds. AP here means `sklearn.metrics.average_precision_score`, a step-weighted precision–recall summary, **not trapezoidal PR-AUC**. Report its fold standard deviation as descriptive variability, not a confidence interval. Ties use lower AP standard deviation, then candidate name.

**Secondary metrics:** precision, recall, F1 and confusion counts. During fold comparison ML cutoffs stay at 0.5; final cutoff optimisation happens only after model selection. Precision with no predicted positives is reported as 0. Recall/F1 with no observed positives and AP without both classes are reported as undefined. Group support must accompany scores.

**Baselines:** always predict no surge; and predict next-week surge when this week's observed price increase is already at least the fixed surge threshold. The latter's AP uses current percentage change as its ranking score. Constant-score AP equals prevalence when both classes exist.

**Measurable criteria:** (1) selected model's mean fold AP exceeds both baselines; (2) final test AP exceeds both test baselines; (3) final test F1 exceeds both baseline F1 scores. Show which criteria are met, even when the model loses. These are relative technical benchmarks, not promised numeric scores or evidence of economic impact. Decide any stakeholder minimum recall/precision before accessing test results.


In [ ]:
# Read metadata only: no test rows or labels are loaded here.
experiment_settings = {}
for name in FOLDS + ["holdout"]:
    path = cfg.PROJECT_ROOT / "experiments" / name / "05_modelling" / "experiment_settings.json"
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}. Run python prepare.py --experiment all first.")
    experiment_settings[name] = json.loads(path.read_text())

reference = experiment_settings["holdout"]
# Learned exclusions/bounds may differ; the target, source snapshot and shared policies must agree.
variable_keys = {"experiment", "train_end", "validation_end", "test_end", "evaluation_end",
                 "apply_documented_exclusions"}
shared_keys = set(cfg.PREPARATION_SETTINGS) - variable_keys
for name, settings in experiment_settings.items():
    preparation = settings["preparation"]
    if any(preparation[key] != cfg.PREPARATION_SETTINGS[key] for key in shared_keys):
        raise ValueError(f"{name}: shared preparation settings changed. Rerun preparation.")
    if preparation["source_manifest_sha256"] != reference["preparation"]["source_manifest_sha256"]:
        raise ValueError("Experiments use different source snapshots. Rerun preparation consistently.")
    for key in ["feature_columns", "numeric_features", "categorical_features", "target_status", "surge_threshold_pct"]:
        if settings[key] != reference[key]:
            raise ValueError(f"{name}: incompatible {key}.")
    if name in FOLDS and preparation["apply_documented_exclusions"]:
        raise ValueError("Early folds must retain retrospective manual exceptions whose review dates are unknown.")

SURGE_THRESHOLD = reference["surge_threshold_pct"]
display(pd.DataFrame([
    {"experiment": name, "train_end": s["train_end"], "validation_end": s["validation_end"],
     "test_end": s["test_end"], "manual_exclusions": s["review_policy"]["documented_training_records_excluded"]}
    for name, s in experiment_settings.items()
]))


## 3. Compare candidates on expanding folds

Each fit gets a **new** median imputer, missingness indicators and categorical encoder trained on that fold alone. Logistic Regression additionally scales numeric features using training statistics. Missing categorical values use a fixed token; unseen categories are ignored by one-hot encoding. Numeric features entirely missing in training are retained by the imputer. Item codes are categories.

Class weighting is derived from training labels only. No evaluation labels influence training weights, preprocessing, tree counts or early stopping. Validation inputs may contain price history observed by their own prediction dates, as intended for weekly prediction. Only one fold is held in memory at a time. The preparation policy intentionally keeps the retrospective manual exceptions in the early folds.


In [ ]:
fold_rows = []
for fold in FOLDS:
    folder, settings, frames = load_experiment(cfg.PROJECT_ROOT, fold, cfg.EXPERIMENTS[fold])
    train, validation = frames["train"], frames["validation"]
    y_train = train.surge_label.astype(int)
    y_validation = validation.surge_label.astype(int)
    if y_validation.nunique() != 2:
        raise ValueError(f"{fold}: AP comparison needs both validation classes; inspect the label audits.")
    X_train = model_inputs(train, settings)
    X_validation = model_inputs(validation, settings)
    print(f"{fold}: {len(train):,} training rows; {len(validation):,} validation rows")

    for name, (scores, predictions) in baseline_scores(validation, SURGE_THRESHOLD).items():
        fold_rows.append({"fold": fold, "candidate": name, "family": "Baseline", "fit_seconds": 0.0,
                          **evaluate(y_validation, scores, predictions)})
    for name, (family, parameters) in CANDIDATES.items():
        pipeline = build_pipeline(family, parameters, settings, y_train,
                                  seed=SEED, n_jobs=N_JOBS, trees=TREES)
        started = perf_counter()
        pipeline.fit(X_train, y_train)
        seconds = perf_counter() - started
        scores = pipeline.predict_proba(X_validation)[:, 1]
        metrics = evaluate(y_validation, scores, scores >= 0.5)
        fold_rows.append({"fold": fold, "candidate": name, "family": family,
                          "fit_seconds": seconds, **metrics})
        print(f"  {name}: AP={metrics['average_precision']:.4f}; F1@0.5={metrics['f1']:.4f}")
        del pipeline
    del frames, train, validation, X_train, X_validation

fold_metrics = pd.DataFrame(fold_rows)
fold_metrics.to_csv(OUTPUT_DIR / "fold_metrics.csv", index=False)
fold_summary = fold_metrics.groupby(["candidate", "family"]).agg(
    mean_ap=("average_precision", "mean"), std_ap=("average_precision", "std"),
    scored_folds=("average_precision", "count"), mean_f1=("f1", "mean"),
    mean_recall=("recall", "mean"), mean_precision=("precision", "mean"),
    total_fit_seconds=("fit_seconds", "sum"),
).reset_index().sort_values(["mean_ap", "std_ap", "candidate"], ascending=[False, True, True])
fold_summary.to_csv(OUTPUT_DIR / "fold_summary.csv", index=False)
display(fold_summary)


## 4. Freeze the candidate, then use the 2026 validation period

Choose the strongest ML candidate by the prespecified fold rule; the benchmark checks can still show that a simple baseline is better. Fit that candidate through December 2025; do not fit any part of it on validation/test rows. Choose the **highest score cutoff maximising validation F1** (conservative tie-break). Validation F1 after cutoff tuning is a selection result, not an unbiased performance estimate.

The fitted pipeline is kept unchanged for final test evaluation and later predictions. We do not refit on train+validation, because that would change the score scale after choosing the cutoff. A future deployment refit needs its own later calibration/evaluation protocol.


In [ ]:
ranked = fold_summary.loc[fold_summary.candidate.isin(CANDIDATES)].copy()
if not ranked.scored_folds.eq(len(FOLDS)).all():
    raise ValueError("Every candidate must have AP from all folds; do not select from inconsistent subsets.")
selected_name = ranked.iloc[0].candidate
selected_family, selected_parameters = CANDIDATES[selected_name]
folder, settings, frames = load_experiment(cfg.PROJECT_ROOT, "holdout", cfg.EXPERIMENTS["holdout"])
train, validation = frames["train"], frames["validation"]
final_pipeline = build_pipeline(selected_family, selected_parameters, settings, train.surge_label.astype(int),
                                seed=SEED, n_jobs=N_JOBS, trees=TREES)
final_pipeline.fit(model_inputs(train, settings), train.surge_label.astype(int))
validation_scores = final_pipeline.predict_proba(model_inputs(validation, settings))[:, 1]
score_cutoff = choose_cutoff(validation.surge_label.astype(int), validation_scores)
print("Selected:", selected_name, "| Validation-selected score cutoff:", score_cutoff)

validation_rows = [
    {"model": selected_name, "cutoff": score_cutoff,
     **evaluate(validation.surge_label, validation_scores, validation_scores >= score_cutoff)},
    {"model": selected_name + " @0.5", "cutoff": 0.5,
     **evaluate(validation.surge_label, validation_scores, validation_scores >= 0.5)},
]
for name, (scores, predictions) in baseline_scores(validation, SURGE_THRESHOLD).items():
    validation_rows.append({"model": name, **evaluate(validation.surge_label, scores, predictions)})
validation_metrics = pd.DataFrame(validation_rows)
validation_metrics.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)
display(validation_metrics)

# Persist the chosen model and decision BEFORE accessing test rows.
joblib.dump(final_pipeline, OUTPUT_DIR / "best_model.joblib")
selection = {"candidate": selected_name, "family": selected_family, "parameters": selected_parameters,
             "score_cutoff": score_cutoff, "selection_metric": "mean expanding-fold average precision",
             "cutoff_rule": "maximum validation F1; highest cutoff breaks ties",
             "seed": SEED, "n_jobs": N_JOBS, "tree_iterations": TREES,
             "feature_columns": settings["feature_columns"], "target": settings["target"],
             "surge_threshold_pct": SURGE_THRESHOLD, "target_status": settings["target_status"],
             "score_interpretation": "uncalibrated model score; not verified event probability",
             "training_end": settings["train_end"], "validation_end": settings["validation_end"],
             "test_end": settings["test_end"], "refit_on_validation": False}
(OUTPUT_DIR / "selection.json").write_text(json.dumps(selection, indent=2))


## 5. Final test — evaluate the frozen choice once

**Only now load April–June 2026 test rows.** No model, feature, threshold or cleaning change follows from these results. If results are weak, report that finding; further development needs a fresh untouched evaluation period. Repeatedly tuning after inspecting this section invalidates the final-test claim.

We report the selected cutoff, the fixed 0.5 reference and both baselines. AP assesses ranking independently of the classification cutoff. Scores from weighted models are not calibrated probabilities.


In [ ]:
test = check_frame(pd.read_parquet(folder / "test.parquet"), settings, "test")
test_scores = final_pipeline.predict_proba(model_inputs(test, settings))[:, 1]
test_predictions = (test_scores >= score_cutoff).astype(int)
test_rows = [
    {"model": selected_name, "cutoff": score_cutoff, **evaluate(test.surge_label, test_scores, test_predictions)},
    {"model": selected_name + " @0.5", "cutoff": 0.5, **evaluate(test.surge_label, test_scores, test_scores >= 0.5)},
]
for name, (scores, predictions) in baseline_scores(test, SURGE_THRESHOLD).items():
    test_rows.append({"model": name, **evaluate(test.surge_label, scores, predictions)})
test_metrics = pd.DataFrame(test_rows)
test_metrics.to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)
display(test_metrics)

selected_mean_ap = float(ranked.iloc[0].mean_ap)
baseline_mean_ap = float(fold_summary.loc[fold_summary.family.eq("Baseline"), "mean_ap"].max())
baseline_test = test_metrics.loc[test_metrics.model.isin(["Always no surge", "Current-change rule"])]
criteria = pd.DataFrame([
    {"criterion": "Mean fold AP exceeds both baselines", "model_value": selected_mean_ap,
     "reference_value": baseline_mean_ap},
    {"criterion": "Test AP exceeds both baselines", "model_value": test_metrics.iloc[0].average_precision,
     "reference_value": baseline_test.average_precision.max()},
    {"criterion": "Test F1 exceeds both baselines", "model_value": test_metrics.iloc[0].f1,
     "reference_value": baseline_test.f1.max()},
])
criteria["assessable"] = criteria[["model_value", "reference_value"]].notna().all(axis=1)
criteria["met"] = pd.Series(pd.NA, index=criteria.index, dtype="boolean")
criteria.loc[criteria.assessable, "met"] = (
    criteria.loc[criteria.assessable, "model_value"] > criteria.loc[criteria.assessable, "reference_value"])
criteria.to_csv(OUTPUT_DIR / "benchmark_checks.csv", index=False)
display(criteria)


## 6. Check food groups, geography and errors

Overall performance can hide weak groups or be dominated by vegetables. Export support, prevalence and confusion counts alongside metrics for each food group, state and district. Undefined group metrics stay missing; zero observed surges cannot establish positive-case detection quality. These are descriptive test breakdowns, not a new tuning step or proof of geographical fairness.


In [ ]:
group_rows = []
for model_name, scores, predictions in [(selected_name, test_scores, test_predictions)] + [
    (name, scores, predictions) for name, (scores, predictions) in baseline_scores(test, SURGE_THRESHOLD).items()
]:
    for level in [["project_group"], ["state"], ["state", "district"]]:
        table = grouped_metrics(test, scores, predictions, level)
        table.insert(0, "model", model_name)
        table.insert(1, "breakdown", "/".join(level))
        group_rows.append(table)
group_results = pd.concat(group_rows, ignore_index=True)
group_results.to_csv(OUTPUT_DIR / "test_group_metrics.csv", index=False)
display(group_results.loc[group_results.breakdown.eq("project_group")])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(test.surge_label, test_predictions, labels=[0, 1],
    display_labels=["No surge", "Surge"], ax=axes[0], colorbar=False)
axes[0].set_title("Frozen model and validation-selected cutoff")
if test.surge_label.nunique() == 2:
    PrecisionRecallDisplay.from_predictions(test.surge_label, test_scores, name=selected_name, ax=axes[1])
    current_scores, _ = baseline_scores(test, SURGE_THRESHOLD)["Current-change rule"]
    PrecisionRecallDisplay.from_predictions(test.surge_label, current_scores, name="Current-change rule", ax=axes[1])
    axes[1].axhline(test.surge_label.mean(), linestyle="--", color="grey", label="Prevalence reference")
    axes[1].legend()
else:
    axes[1].text(.1, .5, "PR comparison undefined: test has one class")
axes[1].set_title("Final-test precision–recall")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "test_evaluation.png", dpi=160, bbox_inches="tight")
plt.show()


## 7. Save auditable predictions and model provenance

Keep identifiers, dates, labels and scores so false alarms/misses can be inspected without retraining. Latest predictions use the same fitted pipeline and cutoff. These historical-snapshot scores do not imply a live dashboard.

The manifest records prepared-input, code and saved-model hashes alongside package versions. Later SHAP work should load **this saved pipeline** and its training-derived preprocessing, then precompute selected explanations for responsive dashboard display. Isolation Forest remains a separate task. Load only trusted model files; joblib stores executable Python objects.


In [ ]:
identifiers = ["item_code", "item", "unit", "project_group", "state", "district", "premise_type", "week_start", "next_week_end"]
for name, frame, scores in [("validation", validation, validation_scores), ("test", test, test_scores)]:
    predictions = frame[identifiers + ["surge_label"]].copy()
    predictions["model_score"] = scores
    predictions["predicted_surge"] = (scores >= score_cutoff).astype(int)
    predictions.to_parquet(OUTPUT_DIR / f"{name}_predictions.parquet", index=False)

latest = pd.read_parquet(folder / "latest_prediction_inputs.parquet")
latest_predictions = latest[identifiers].copy()
if len(latest):
    latest_predictions["model_score"] = final_pipeline.predict_proba(model_inputs(latest, settings))[:, 1]
    latest_predictions["predicted_surge"] = (latest_predictions.model_score >= score_cutoff).astype(int)
else:
    latest_predictions["model_score"] = pd.Series(dtype=float)
    latest_predictions["predicted_surge"] = pd.Series(dtype=int)
    print("No supported latest-week rows. No stale local predictions substituted.")
latest_predictions.to_parquet(OUTPUT_DIR / "latest_predictions.parquet", index=False)

input_hashes = {}
for name in FOLDS + ["holdout"]:
    input_folder = cfg.PROJECT_ROOT / "experiments" / name / "05_modelling"
    files = ["experiment_settings.json", "train.parquet", "validation.parquet"]
    if name == "holdout":
        files += ["test.parquet", "latest_prediction_inputs.parquet"]
    for filename in files:
        with (input_folder / filename).open("rb") as handle:
            input_hashes[f"{name}/{filename}"] = hashlib.file_digest(handle, "sha256").hexdigest()
versions = {}
for package in ["pandas", "numpy", "scikit-learn", "xgboost", "xgboost-cpu", "lightgbm", "joblib"]:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        pass  # Only one of xgboost / xgboost-cpu needs to be installed.
code_hashes = {name: hashlib.sha256((Path(cfg.__file__).parent / name).read_bytes()).hexdigest()
               for name in ["06_model_training_and_evaluation.ipynb", "pricecatcher_modelling.py", "pricecatcher_config.py"]}
with (OUTPUT_DIR / "best_model.joblib").open("rb") as handle:
    model_hash = hashlib.file_digest(handle, "sha256").hexdigest()
manifest = {"code_sha256": code_hashes, "model_sha256": model_hash, "selection": selection, "candidates": CANDIDATES, "folds": FOLDS,
            "input_sha256": input_hashes, "package_versions": versions,
            "holdout_preparation": settings["preparation"], "holdout_cleaning": settings["review_policy"],
            "metric_definition": "sklearn average precision; not trapezoidal PR-AUC",
            "limitations": ["Uncalibrated scores", "Observed-median labels can reflect monitoring composition",
                            "Per-group scarcity", "No demonstrated affordability or adoption outcomes"]}
(OUTPUT_DIR / "model_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Saved model, results and prediction files to:", OUTPUT_DIR)


## What to write in the report

Report the fold dates, candidate settings, AP definition, preprocessing/weighting policy, selected candidate and validation cutoff. Present fold variability, final-test baseline comparisons, confusion counts and food-group results. State which prespecified criteria were met; do not call the integration of existing algorithms inherently novel or universally best.

The fold results support candidate selection; validation results after cutoff tuning are not final generalisation evidence. The test is the final estimate for the configured period, subject to temporal dependence and source uncertainty. No affordability, SDG or behaviour-change effect was measured. Future notebooks can implement separate anomaly detection, SHAP precomputation and the prototype dashboard without using test outcomes to retune this experiment.
